In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer



In [48]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head(10)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
5,spam,FreeMsg Hey there darling it's been 3 week's n...,NaN,NaN,NaN
6,ham,Even my brother is not like to speak with me. ...,NaN,NaN,NaN
7,ham,As per your request 'Melle Melle (Oru Minnamin...,NaN,NaN,NaN
8,spam,WINNER!! As a valued network customer you have...,NaN,NaN,NaN
9,spam,Had your mobile 11 months or more? U R entitle...,NaN,NaN,NaN


In [50]:
len(df)

5572

In [49]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [2]:
nltk.download("stopwords")
nltk.download("wordnet")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


In [4]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)


In [38]:
df["text"] = df["v2"]
df["text"] = df["text"].apply(preprocess_text)

X = df["text"]
y = df["v1"]

In [39]:
from sklearn.model_selection import train_test_split

# Train-test split


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorization


vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [52]:
print(X_train_vec[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (1, 15000)>
  Coords	Values
  (0, 7922)	0.26278103575259915
  (0, 12017)	0.34916377791000774
  (0, 8462)	0.2896672176554671
  (0, 1360)	0.4476174651248852
  (0, 7972)	0.5122315352334892
  (0, 8471)	0.5122315352334892


In [43]:
from sklearn.naive_bayes import MultinomialNB

# NAIVE BAYES

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)


MultinomialNB()

In [44]:
nb_pred = nb_model.predict(X_test_vec)


In [45]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, nb_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, nb_pred))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, nb_pred))


===== NAIVE BAYES RESULTS =====
Accuracy: 0.9632286995515695

Classification Report:

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115

Confusion Matrix:

[[966   0]
 [ 41 108]]


In [46]:
from sklearn.svm import LinearSVC


svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)


LinearSVC()

In [47]:
svm_pred = svm_model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, svm_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, svm_pred))
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, svm_pred))


===== SVM RESULTS =====
Accuracy: 0.9856502242152466

Classification Report:

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.90      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

Confusion Matrix:

[[965   1]
 [ 15 134]]
